# 00 · Setup and smoke test

Goal: check that the environment works, the raw files are complete and match the competition description,
and that `src/cv.py` (folds, metric, results logging) behaves as expected. No modelling, no plots.

In [1]:
import sys, time, importlib
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import cv

cv.set_seed()
cv.ensure_dirs()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

print("Python", sys.version.split()[0])
for m in ["pandas", "numpy", "scipy", "matplotlib", "seaborn", "sklearn", "statsmodels",
          "lightgbm", "xgboost", "catboost", "optuna", "shap", "pyarrow"]:
    print(f"{m:12s}", importlib.import_module(m).__version__)
print("ROOT:", ROOT)

Python 3.12.7
pandas       3.1.1
numpy        2.5.4
scipy        1.18.1
matplotlib   3.11.2
seaborn      0.13.2
sklearn      1.9.1
statsmodels  0.15.0
lightgbm     4.7.0
xgboost      3.4.1
catboost     1.2.10


/home/fsa/Desktop/projects/Store Sales/store-sales/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


optuna       5.0.0
shap         0.54.1
pyarrow      26.0.1
ROOT: /home/fsa/Desktop/projects/Store Sales/store-sales


Every library is on the pinned version. Python is 3.12.7 here and the files were tested on 3.12.3; a patch-level difference doesn't change anything. The tqdm warning comes from optuna or shap importing tqdm while ipywidgets isn't installed. It's harmless (progress bars just print as text), but I added ipywidgets to requirements.txt so it goes away.

In [2]:
# All raw files present, with their sizes
for name, fname in cv.RAW_FILES.items():
    p = cv.DATA_RAW / fname
    print(f"{fname:25s}", f"{p.stat().st_size / 1e6:8.1f} MB" if p.exists() else "MISSING")
assert all((cv.DATA_RAW / f).exists() for f in cv.RAW_FILES.values()), "download the data first"

train.csv                    121.8 MB
test.csv                       1.0 MB
stores.csv                     0.0 MB
oil.csv                        0.0 MB
holidays_events.csv            0.0 MB
transactions.csv               1.6 MB
sample_submission.csv          0.3 MB


All seven files downloaded into data/raw.

In [3]:
# Load every file with the dtypes defined in cv.read_raw
t0 = time.perf_counter()
raw = {name: cv.read_raw(name) for name in cv.RAW_FILES}
print(f"loaded in {time.perf_counter() - t0:.1f}s\n")
for name, df in raw.items():
    print(f"{name:18s} {str(df.shape):16s} {df.memory_usage(deep=True).sum() / 1e6:7.1f} MB")
train, test = raw["train"], raw["test"]
train.head()

loaded in 0.8s

train              (3000888, 6)       131.3 MB
test               (28512, 5)           1.0 MB
stores             (54, 5)              0.0 MB
oil                (1218, 2)            0.0 MB
holidays           (350, 6)             0.0 MB
transactions       (83488, 3)           1.1 MB
sample_submission  (28512, 2)           0.2 MB


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


Loading everything takes under a second, and train fits in about 131 MB with the compact dtypes, so I can keep the whole dataset in memory without chunking.

transactions has 83,488 rows, fewer than 54 stores × 1,684 days = 90,936, so about 7,400 store-days have no transaction count. My guess is stores that opened late or were closed on some days; I'll check that in EDA.

The first rows are all zero sales on 2013-01-01. That's New Year's Day, so stores were probably closed, but I'll confirm it in EDA rather than assume it.

In [4]:
print(train.dtypes, "\n")
print(test.dtypes)

id                      int32
date           datetime64[us]
store_nbr                int8
family                    str
sales                 float64
onpromotion             int32
dtype: object 

id                      int32
date           datetime64[us]
store_nbr                int8
family                    str
onpromotion             int32
dtype: object


date is datetime64[us] (pandas 3 uses microsecond resolution by default) and family uses the new str dtype. sales is float, not int, because some families are sold by weight, so fractional values are real and not a data error.

In [5]:
# Date ranges must match the competition description
def date_range(df):
    return df["date"].min().date(), df["date"].max().date(), df["date"].nunique()

print("train:", date_range(train))
print("test: ", date_range(test))
assert train["date"].min() == cv.TRAIN_START and train["date"].max() == cv.TRAIN_END
assert test["date"].min() == cv.TEST_START and test["date"].max() == cv.TEST_END
assert test["date"].nunique() == cv.HORIZON

train: (datetime.date(2013, 1, 1), datetime.date(2017, 8, 15), 1684)
test:  (datetime.date(2017, 8, 16), datetime.date(2017, 8, 31), 16)


The dates match the competition description: train runs from 2013-01-01 to 2017-08-15, and test is exactly 16 days, 2017-08-16 to 2017-08-31.

In [6]:
# Number of series and whether every date has every series
n_stores, n_fam = train["store_nbr"].nunique(), train["family"].nunique()
n_series = train.groupby(cv.KEYS, observed=True).ngroups
print(f"stores={n_stores}  families={n_fam}  series={n_series}  (stores x families = {n_stores * n_fam})")

rows_per_date = train.groupby("date").size()
print("rows per date:", rows_per_date.value_counts().to_dict())

all_days = pd.date_range(cv.TRAIN_START, cv.TRAIN_END, freq="D")
missing_days = all_days.difference(pd.DatetimeIndex(train["date"].unique()))
print(f"calendar days={len(all_days)}  days in train={train['date'].nunique()}  missing={len(missing_days)}")
print("missing days:", [d.date() for d in missing_days])

stores=54  families=33  series=1782  (stores x families = 1782)
rows per date: {1782: 1684}
calendar days=1688  days in train=1684  missing=4
missing days: [datetime.date(2013, 12, 25), datetime.date(2014, 12, 25), datetime.date(2015, 12, 25), datetime.date(2016, 12, 25)]


There are 1,782 series (54 stores × 33 families), and every date in train has all 1,782 rows, so the panel has no holes in it. The only dates missing are the four Christmas Days.

That has a direct consequence for features. If I build lags by shifting rows inside each series, "7 rows back" is not "7 days back" whenever a window crosses December 25. So in Phase 2, lags and rolling windows have to be computed on the calendar (reindex to a full daily calendar first, or shift by date), not by row position.

A complete panel also doesn't mean every value is a real observation. A zero can mean "sold nothing" or "store not open yet", and those need different treatment.

In [7]:
# Test structure: every series once per test day, ids continue after train, matches sample_submission
sub = raw["sample_submission"]
print("test rows:", len(test), " expected:", n_series * cv.HORIZON)
print("test series == train series:",
      set(map(tuple, test[cv.KEYS].drop_duplicates().to_numpy())) == set(map(tuple, train[cv.KEYS].drop_duplicates().to_numpy())))
print("test ids:", test["id"].min(), "->", test["id"].max(), " train last id:", train["id"].max())
print("sample_submission ids == test ids:", sub["id"].equals(test["id"]))
assert len(test) == n_series * cv.HORIZON

test rows: 28512  expected: 28512


test series == train series: True
test ids: 3000888 -> 3029399  train last id: 3000887
sample_submission ids == test ids: True


Test has 28,512 rows = 1,782 series × 16 days, the series are exactly the training series, and the ids continue straight after train. sample_submission is in the same order as test. That means there's no cold-start problem in the test set: every series I have to forecast has history.

In [8]:
# Basic quality checks on train
print("missing values:\n", train.isna().sum().to_string(), "\n")
print("negative sales:", int((train["sales"] < 0).sum()))
print("share of zero sales:", round(float((train["sales"] == 0).mean()), 3))
print("sales max:", float(train["sales"].max()))
print("onpromotion range:", int(train["onpromotion"].min()), "->", int(train["onpromotion"].max()),
      "| test:", int(test["onpromotion"].min()), "->", int(test["onpromotion"].max()))

missing values:
 id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0 

negative sales: 0
share of zero sales: 0.313
sales max: 124717.0
onpromotion range: 0 -> 741 | test: 0 -> 646


No missing values and no negative sales. 31.3% of all rows are zero, which is a big part of why the target is log1p(sales) and the metric is RMSLE: the log compresses the huge sales values, and the +1 keeps zeros defined.

The maximum sale is 124,717, which looks extreme next to typical values. I'll find out which series and date it is in EDA before deciding whether it's an outlier or a genuine event.

onpromotion goes up to 741 in train and 646 in test. So it's the number of items in that family on promotion that day, not a 0/1 flag. I'll treat it as a count (probably log1p of it) in the features.

In [9]:
# Side files: coverage of the train and test periods
oil, tr, hol, stores = raw["oil"], raw["transactions"], raw["holidays"], raw["stores"]

print("oil:          ", date_range(oil), "| missing prices:", int(oil["dcoilwtico"].isna().sum()))
print("transactions: ", date_range(tr), "| covers test period:", tr["date"].max() >= cv.TEST_START)
print("holidays:     ", date_range(hol), "| rows in test period:",
      int(hol["date"].between(cv.TEST_START, cv.TEST_END).sum()))
print("stores:       ", stores.shape, "| store ids match train:",
      set(stores["store_nbr"]) == set(train["store_nbr"]))
print("\nholiday types:", hol["type"].value_counts().to_dict())
print("holiday locales:", hol["locale"].value_counts().to_dict())
print("transferred:", hol["transferred"].value_counts().to_dict())

oil:           (datetime.date(2013, 1, 1), datetime.date(2017, 8, 31), 1218) | missing prices: 43
transactions:  (datetime.date(2013, 1, 1), datetime.date(2017, 8, 15), 1682) | covers test period: False
holidays:      (datetime.date(2012, 3, 2), datetime.date(2017, 12, 26), 312) | rows in test period: 1
stores:        (54, 5) | store ids match train: True

holiday types: {'Holiday': 221, 'Event': 56, 'Additional': 51, 'Transfer': 12, 'Bridge': 5, 'Work Day': 5}
holiday locales: {'National': 174, 'Local': 152, 'Regional': 24}
transferred: {False: 338, True: 12}


oil has 1,218 rows over 1,704 calendar days, so it only has trading days (no weekends), plus 43 missing prices. It needs reindexing to daily and forward filling. It also runs to 2017-08-31, i.e. it covers the test period. Having the column doesn't mean I'd know those prices at forecast time, and that's a question for Phase 2.

transactions stops on 2017-08-15, so there's nothing for the test period. Any feature built from transactions has to be lagged by at least 16 days, or left out. It also has 1,682 dates, two fewer than train, so two days have no transactions recorded for any store. I'll look at which ones.

holidays has 350 rows but only 312 distinct dates, so some dates have more than one event (for example a national and a local one on the same day). If I merge it onto train by date as it is, train rows get duplicated. It has to be aggregated to one row per date and locale before any merge.

There are 12 rows of type Transfer and 12 rows with transferred = True. They line up: a holiday that was moved is marked transferred on its official date, and the day it was actually celebrated appears as a Transfer row. So the effect belongs on the Transfer date, not the original one.

Only one holiday row falls in the test period. I'll check what it is and which city it applies to.

In [10]:
# Backtest folds: defined once in src/cv.py
folds = cv.folds_table()
folds["valid_rows"] = [len(cv.split(train, f)[1]) for f in cv.FOLDS]
folds["train_rows"] = [len(cv.split(train, f)[0]) for f in cv.FOLDS]
print("test period:", cv.TEST_START.date(), cv.TEST_START.day_name(), "->", cv.TEST_END.date(), cv.TEST_END.day_name())
folds

test period: 2017-08-16 Wednesday -> 2017-08-31 Thursday


,fold,train_end,valid_start,valid_start_day,valid_end,valid_end_day,valid_rows,train_rows
0,1,2017-06-12,2017-06-13,Tuesday,2017-06-28,Wednesday,28512,2886840
1,2,2017-06-28,2017-06-29,Thursday,2017-07-14,Friday,28512,2915352
2,3,2017-07-14,2017-07-15,Saturday,2017-07-30,Sunday,28512,2943864
3,4,2017-07-30,2017-07-31,Monday,2017-08-15,Tuesday,28512,2972376


The four folds are what I planned: 16-day windows back to back from 2017-06-13 to 2017-08-15, each with 28,512 rows to predict and an expanding training set before it. Fold 4 ends on the last training day, right before the test period.

The test period starts on a Wednesday, while the folds start on Tuesday, Thursday, Saturday and Monday. For the overall score this doesn't matter. For the "day 1 vs day 16" error analysis it does, because horizon day 1 is a different weekday in each fold. I'll come back to it in Phase 3.

Fold 4 also contains August 10, Ecuador's independence holiday, so that fold includes a holiday effect. I'll check in EDA whether it was moved in 2017.

In [11]:
# Fold sanity: each window is 16 days, folds don't overlap, train always ends before its window
for f in cv.FOLDS:
    tr_f, va_f = cv.split(train, f)
    assert va_f["date"].nunique() == cv.HORIZON
    assert tr_f["date"].max() < va_f["date"].min()
assert all(a.valid_end < b.valid_start for a, b in zip(cv.FOLDS, cv.FOLDS[1:]))
assert cv.FOLDS[-1].valid_end == cv.TRAIN_END
print("folds OK")

folds OK


Each window has exactly 16 dates, training always ends before its window, the windows don't overlap, and the last one ends on 2017-08-15.

In [12]:
# Metric checks against hand-computed values
assert cv.rmsle([0, 1, 10], [0, 1, 10]) == 0.0
y, p = np.array([0.0, 3.0]), np.array([1.0, 1.0])
manual = np.sqrt(((np.log(2) - np.log(1)) ** 2 + (np.log(2) - np.log(4)) ** 2) / 2)
assert np.isclose(cv.rmsle(y, p), manual)
assert cv.rmsle([5.0], [-3.0]) == cv.rmsle([5.0], [0.0])   # negative predictions are clipped to 0
print("rmsle OK, example:", round(cv.rmsle(y, p), 5))

rmsle OK, example: 0.69315


The metric matches the hand calculation. The example comes out at 0.69315 = ln 2, which makes sense: both predictions are off by exactly a factor of 2 on the log1p scale (log 2 vs log 1, and log 2 vs log 4). Negative predictions are clipped to 0, the same way the submission will be.

In [13]:
# evaluate / log_result / save round trip, written to a temporary folder (results.csv is untouched)
import tempfile
tmp = Path(tempfile.mkdtemp())

# a dummy model: constant prediction of 1.0 on every validation row
dummy = pd.concat(
    [cv.split(train, f)[1][["date", *cv.KEYS]].assign(fold=f.fold, y_pred=1.0) for f in cv.FOLDS],
    ignore_index=True,
)
scores = cv.evaluate(dummy, train)
print("dummy fold scores:", {k: round(v, 4) for k, v in scores.items()})

cv.log_result("dummy_a", scores, params="c=1", notes="smoke test", path=tmp / "results.csv")
cv.log_result("dummy_b", {k: v + 0.1 for k, v in scores.items()}, path=tmp / "results.csv")
cv.log_result("dummy_a", scores, notes="re-run replaces the row", path=tmp / "results.csv")
res = cv.show_results(tmp / "results.csv")
assert len(res) == 2 and res.loc[0, "model"] == "dummy_a" and res.loc[0, "notes"] == "re-run replaces the row"
assert res.loc[1, "notes"] == ""                  # empty text stays empty, not NaN

(tmp / "empty.csv").touch()                       # an empty file behaves like a missing one
assert len(cv.log_result("x", scores, path=tmp / "empty.csv")) == 1

out = cv.save_backtest_preds("dummy_a", dummy, preds_dir=tmp)
back = cv.load_backtest_preds("dummy_a", preds_dir=tmp)
assert len(back) == len(dummy) and cv.evaluate(back, train) == scores

try:                                               # a model that drops rows must fail loudly
    cv.evaluate(dummy.iloc[:-5], train)
    raise AssertionError("evaluate should have failed")
except ValueError as e:
    print("missing rows caught:", e)
res

dummy fold scores: {1: 3.8325, 2: 3.8733, 3: 3.8729, 4: 3.8709}
missing rows caught: fold 4: expected 28512 predictions, got 28507, 5 validation rows unmatched


,model,mean_rmsle,std_rmsle,fold_1,fold_2,fold_3,fold_4,params,notes,logged_at
0,dummy_a,3.86239,0.01994,3.83252,3.87326,3.87287,3.8709,,re-run replaces the row,2026-11-14 01:42
1,dummy_b,3.96239,0.01994,3.93252,3.97326,3.97287,3.9709,,,2026-11-14 01:42


The dummy model (always predict 1) scores about 3.86, with very little difference between folds. It's only here to test the mechanics, not as a baseline; real sales are mostly in the hundreds, so predicting 1 is far off.

The mechanics all work. Re-running a model replaced its row instead of adding a second one, empty notes stayed empty instead of turning into NaN, an empty results file behaved like a missing one, and the saved predictions loaded back with identical scores. When I dropped five rows from the predictions, evaluate refused to score them and said exactly how many were missing.

In [14]:
print("results.csv exists:", cv.RESULTS_PATH.exists())
print("SMOKE TEST PASSED")

results.csv exists: False
SMOKE TEST PASSED


Everything passed, and the real results.csv was never created: the test only wrote to a temporary folder.

## Where I am

The environment is set up on the pinned versions, the data is downloaded and read-only, and the backtest, metric and results logging in `src/cv.py` all work on the real data. The data matches the competition description: 3,000,888 training rows, 1,782 complete series from 2013-01-01 to 2017-08-15, and a 16-day test period with no new series. I haven't fitted any model yet. The four backtest folds are fixed and every model from Phase 3 onward will be scored on them.

The smoke test already gave me things to carry forward. Christmas Day is missing every year, so lags must be built on dates, not row positions. Zeros make up almost a third of the data, which supports log1p and RMSLE. onpromotion is a count, not a flag. transactions stops before the test period. holidays has several events on some dates, so it would duplicate rows if merged as is. And oil only covers trading days and reaches into the test period, which raises the question of whether I could really know those prices in advance.

Next steps: Phase 1, the time series EDA. I'll look at total sales over time (trend, weekly and yearly patterns, holidays, paydays, the April 2016 earthquake), sales by family and by store, which series are always zero and which stores open late, the 124,717 maximum, the effects of promotions, oil and transactions, and then STL decomposition, stationarity tests and ACF/PACF. It ends with a findings → decisions list.